# 06. Dynamic Resource Scheduler & Sampling Control
Demonstrates `ResourceScheduler`'s direction-weighted batch sampling (per `configs/training/multilingual.yaml`) and `MultilingualPairMixer`'s weighted-mixture dataset construction.

In [ ]:
# ============================================================
# PATH BOOSTER — Guarantees project root in sys.path & CWD
# ============================================================
import os, sys
try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)
proj_dir = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(proj_dir):
    os.chdir(proj_dir)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())


In [1]:
import os

if "COLAB_GPU" in os.environ or os.environ.get("COLAB_RELEASE_TAG"):
    if not os.path.exists("Ekegusii-LLM-Translation"):
        os.system("git clone https://github.com/aykahsay/Ekegusii-LLM-Translation.git")
    os.chdir("Ekegusii-LLM-Translation")
    os.system("pip install -q -r requirements.txt")
elif not os.path.exists("src") and os.path.basename(os.getcwd()) == "notebooks":
    # Running locally via `jupyter nbconvert` from within notebooks/ --
    # the repo root (containing src/, data/) is one directory up.
    os.chdir("..")

import sys
sys.path.insert(0, os.getcwd())

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")


In [2]:
import os, sys
p = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(p) and p not in sys.path: sys.path.insert(0, p); os.chdir(p)

from omegaconf import OmegaConf
from src.master_corpus.manager import MasterCorpusManager
from src.master_corpus.scheduler import ResourceScheduler
from src.task_generation.translation_pairs import direction_counts
from src.task_generation.multilingual_pairs import MultilingualPairMixer
from src.utils.constants import LANGUAGE_CODES

manager = MasterCorpusManager()
train_df = manager.load_train_split()
multilingual_cfg = OmegaConf.load('configs/training/multilingual.yaml')
multilingual_cfg

INFO | Loaded dataset split [master_train.csv]: 39,421 rows.


{'languages': ['eng', 'swh', 'eke'], 'direction_weights': {'eng_to_eke': 1.0, 'eke_to_eng': 1.0, 'swa_to_eke': 1.0, 'eke_to_swa': 1.0, 'eng_to_swa': 0.5, 'swa_to_eng': 0.5}, 'sampling_temperature': 1.5, 'lexical_augmentation': {'enabled': False, 'mix_ratio': 0.1}}

## Per-direction sampling probabilities

In [3]:
counts = direction_counts(train_df)
counts_by_key = {
    f'{LANGUAGE_CODES[src]}_to_{LANGUAGE_CODES[tgt]}': count
    for label, count in counts
    for src, tgt in [label.split('->')]
}
scheduler = ResourceScheduler(multilingual_cfg)
probs = scheduler.compute_direction_sampling_probs(counts_by_key)
probs

INFO | Extracted 37,721 'English'->'Ekegusii' pairs from 39,421 rows.


INFO | Extracted 37,721 'Ekegusii'->'English' pairs from 39,421 rows.


INFO | Extracted 27,092 'Kiswahili'->'Ekegusii' pairs from 39,421 rows.


INFO | Extracted 27,092 'Ekegusii'->'Kiswahili' pairs from 39,421 rows.


INFO | Extracted 28,792 'English'->'Kiswahili' pairs from 39,421 rows.


INFO | Extracted 28,792 'Kiswahili'->'English' pairs from 39,421 rows.


INFO | Computed sampling probabilities (temperature=1.5): {'eng_to_eke': 0.22526608467210832, 'eke_to_eng': 0.22526608467210832, 'swa_to_eke': 0.180662378496575, 'eke_to_swa': 0.180662378496575, 'eng_to_swa': 0.09407153683131678, 'swa_to_eng': 0.09407153683131678}


{'eng_to_eke': 0.22526608467210832,
 'eke_to_eng': 0.22526608467210832,
 'swa_to_eke': 0.180662378496575,
 'eke_to_swa': 0.180662378496575,
 'eng_to_swa': 0.09407153683131678,
 'swa_to_eng': 0.09407153683131678}

## Simulated batch quota (batch_size=64)

In [4]:
scheduler.build_mixed_batch_plan(counts_by_key, batch_size=64)

INFO | Computed sampling probabilities (temperature=1.5): {'eng_to_eke': 0.22526608467210832, 'eke_to_eng': 0.22526608467210832, 'swa_to_eke': 0.180662378496575, 'eke_to_swa': 0.180662378496575, 'eng_to_swa': 0.09407153683131678, 'swa_to_eng': 0.09407153683131678}


{'eng_to_eke': 17,
 'eke_to_eng': 13,
 'swa_to_eke': 14,
 'eke_to_swa': 11,
 'eng_to_swa': 6,
 'swa_to_eng': 3}

## Weighted-mixture dataset (sample)

In [5]:
sample_df = train_df.sample(500, random_state=42)
mixer = MultilingualPairMixer(multilingual_cfg)
mixture = mixer.build_weighted_mixture(sample_df, target_total=200)
mixture['source_lang'].str.cat(mixture['target_lang'], sep='->').value_counts()

INFO | Extracted 476 'English'->'Ekegusii' pairs from 500 rows.


INFO | Extracted 476 'Ekegusii'->'English' pairs from 500 rows.


INFO | Extracted 345 'Kiswahili'->'Ekegusii' pairs from 500 rows.


INFO | Extracted 345 'Ekegusii'->'Kiswahili' pairs from 500 rows.


INFO | Extracted 369 'English'->'Kiswahili' pairs from 500 rows.


INFO | Extracted 369 'Kiswahili'->'English' pairs from 500 rows.


INFO | Computed sampling probabilities (temperature=1.5): {'eng_to_eke': 0.2243342186761914, 'eke_to_eng': 0.2243342186761914, 'swa_to_eke': 0.1810104376273729, 'eke_to_swa': 0.1810104376273729, 'eng_to_swa': 0.09465534369643568, 'swa_to_eng': 0.09465534369643568}


INFO | Built weighted mixture: 200 rows (target was 200).


source_lang
English->Ekegusii      51
Ekegusii->Kiswahili    40
Ekegusii->English      33
Kiswahili->Ekegusii    32
Kiswahili->English     23
English->Kiswahili     21
Name: count, dtype: int64